<a href="https://colab.research.google.com/github/jangbu12/mac-237-labs/blob/main/MAC237_LAB1_S01_Risk_And_Threat_Model_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAC 237 Graded Lab 1, Session 1: risk, the CIA triad and a threat model

**Monday 14 September 2026. Room B124, or a browser, whichever you are sitting at.**

This notebook is the graded work for session 1. Every cell uses only the Python 3 standard
library, so it runs unchanged in Google Colab in a browser or in Python 3 on the Kali image
in Room B124. Nothing here needs administrative rights, nothing opens a socket, and nothing
reads a file it did not write.

**Nothing in this notebook contacts any host outside it.** All of the data is built by the
notebook itself. Colab runs on Google's infrastructure and Kali in B124 is on the college
network, and neither is yours to send traffic from.

The rest of the hour in B124 is Kali practice. It is not graded and it is not free time: it
is the shell work that Lab 2 assumes you already have under your fingers. The lab document
names what to practice in each session.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell. The token is printed into the cell output
and the output is what gets saved, so it travels with the notebook into your repository.

This is the first lab session of the semester, so it is also the session in which your
repository comes into existence. Open github.com in another tab, create a **private**
repository named `mac237-labs`, and invite the instructor as a collaborator under Settings,
Collaborators. Everything you produce for this lab goes in a folder named `lab1`. Budget
fifteen minutes and do not rush the private setting, because a repository the instructor
cannot open is a lab that cannot be graded.

In [ ]:
import hashlib, datetime

sid = '34711600'
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC237 Lab 1 token:', TOKEN)

MAC237 Lab 1 token: fb5f5f596938e807


the token of lab 1 after i enter my sid is
 590865060a34a9bf

## Section 2. One small system, three properties

Choose one very small realistic system: your home Wi-Fi router, the college library
laptop-checkout kiosk, or a personal password manager on your laptop. Do not choose a shared
enterprise system, and do not test anything against the system you choose. This section is
analysis, not activity on a device.

The rows below are a worked example for the kiosk. **Replace all three with your own system.**
You are graded on your rows, not on these.

Be specific. A row that reads encryption protects data is worth nothing. The check at the
bottom of the cell refuses a field of fewer than six words, which catches the laziest version
of that mistake and no other. It cannot tell a vague sentence from a sharp one, so that part
is on you.

In [ ]:
SYSTEM = 'my home Wi-Fi router sherpa family '

CIA = {
    'confidentiality': {
        'asset':   'the wireless network credentials and connected device information stored by my router',
        'threat':  'an unauthorized person joins my wireless network and observes private information from connected devices',
        'control': 'WPA3 encryption with a strong unique wireless password protects access to my home network',
    },
    'integrity': {
        'asset':   'the router configuration settings that control wireless access and network security',
        'threat':  'an unauthorized person changes router settings and weakens security or redirects network traffic',
        'control': 'a strong administrator password and disabled remote administration protect router configuration settings',
    },
    'availability': {
        'asset':   'the home internet connection provided through my Wi-Fi router',
        'threat':  'the router stops working and household devices cannot access the internet when needed',
        'control': 'a backup router and saved configuration information provide recovery when the main router fails',
    },
}

FIELDS = ('asset', 'threat', 'control')
PROPS = ('confidentiality', 'integrity', 'availability')

thin = []
for prop in PROPS:
    assert prop in CIA, 'missing row: ' + prop
    for field in FIELDS:
        words = len(CIA[prop].get(field, '').split())
        if words < 6:
            thin.append(f'{prop}.{field} is {words} words, needs at least 6')

print('system:', SYSTEM)
print('token :', TOKEN)
print()
for prop in PROPS:
    print(prop.upper())
    for field in FIELDS:
        print(f'  {field:8s} {CIA[prop][field]}')

assert not thin, '; '.join(thin)
print()
print('  CHECK PASS: three properties, each with an asset, a threat and a control')

system: my home Wi-Fi router sherpa family 
token : fb5f5f596938e807

CONFIDENTIALITY
  asset    the wireless network credentials and connected device information stored by my router
  threat   an unauthorized person joins my wireless network and observes private information from connected devices
  control  WPA3 encryption with a strong unique wireless password protects access to my home network
INTEGRITY
  asset    the router configuration settings that control wireless access and network security
  threat   an unauthorized person changes router settings and weakens security or redirects network traffic
  control  a strong administrator password and disabled remote administration protect router configuration settings
AVAILABILITY
  asset    the home internet connection provided through my Wi-Fi router
  threat   the router stops working and household devices cannot access the internet when needed
  control  a backup router and saved configuration information provide recovery when the

## Section 3. STRIDE, and which risk you would actually work on first

The CIA triad names what you are protecting. STRIDE names what goes wrong. Six categories,
one realistic instance each, for the same system you chose above.

Each row also carries two numbers you assign yourself, from 1 to 5: how likely you think the
instance is in a year, and how much it would hurt if it happened. Their product is the risk
score, which is the crudest useful risk model there is and the one almost every organization
starts with.

Replace these six rows with your own, the same way you replaced the three above.

In [ ]:
# category -> (one sentence instance, the CIA property it attacks, likelihood, impact)
STRIDE = {
    'spoofing': (
        'an unauthorized person uses stolen wireless credentials to connect to my home network while pretending to be an authorized user',
        'confidentiality', 3, 4),

    'tampering': (
        'an unauthorized person changes router security settings after gaining access to the administration interface',
        'integrity', 2, 5),

    'repudiation': (
        'someone changes a router configuration and there is no useful record showing which authorized account made the change',
        'integrity', 2, 3),

    'information disclosure': (
        'an unauthorized person connected to my wireless network observes information about devices communicating across the network',
        'confidentiality', 3, 4),

    'denial of service': (
        'the router becomes unavailable and household devices can no longer connect to the internet',
        'availability', 3, 4),

    'elevation of privilege': (
        'a person with limited network access gains administrative access to the router and controls its security settings',
        'integrity', 2, 5),
}

CATEGORIES = ('spoofing', 'tampering', 'repudiation', 'information disclosure',
              'denial of service', 'elevation of privilege')

for cat in CATEGORIES:
    assert cat in STRIDE, 'missing STRIDE row: ' + cat
    sentence, prop, likelihood, impact = STRIDE[cat]
    assert len(sentence.split()) >= 8, cat + ': the instance is too short to be an instance'
    assert prop in PROPS, cat + ': ' + prop + ' is not one of the three properties'
    assert 1 <= likelihood <= 5 and 1 <= impact <= 5, cat + ': scores run from 1 to 5'

print('token:', TOKEN)
print()
for cat in CATEGORIES:
    sentence, prop, likelihood, impact = STRIDE[cat]
    print(f'{cat:24s} {prop:16s} L={likelihood} I={impact}')
    print(f'{"":24s} {sentence}')

print()
print('  CHECK PASS: six categories, each an instance in the system you chose')

token: fb5f5f596938e807

spoofing                 confidentiality  L=3 I=4
                         an unauthorized person uses stolen wireless credentials to connect to my home network while pretending to be an authorized user
tampering                integrity        L=2 I=5
                         an unauthorized person changes router security settings after gaining access to the administration interface
repudiation              integrity        L=2 I=3
                         someone changes a router configuration and there is no useful record showing which authorized account made the change
information disclosure   confidentiality  L=3 I=4
                         an unauthorized person connected to my wireless network observes information about devices communicating across the network
denial of service        availability     L=3 I=4
                         the router becomes unavailable and household devices can no longer connect to the internet
elevation of privilege   integ

In [ ]:
ranked = sorted(((l * i, cat) for cat, (_, _, l, i) in STRIDE.items()), reverse=True)

print('token:', TOKEN)
print()
print(f'{"rank":>4}  {"score":>5}  category')
for n, (score, cat) in enumerate(ranked, start=1):
    print(f'{n:>4}  {score:>5}  {cat}')

scores = [s for s, _ in ranked]
assert scores == sorted(scores, reverse=True), 'the table is not in rank order'
assert scores[0] == max(l * i for _, _, l, i in STRIDE.values())
assert len(ranked) == 6

top_score, top_cat = ranked[0]
ties = [c for s, c in ranked if s == top_score]
print()
print(f'  highest score: {top_cat} at {top_score}' +
      ('' if len(ties) == 1 else '  (tied with ' + ', '.join(ties[1:]) + ')'))
print('  CHECK PASS: six risks scored and ranked')
print('  token:', TOKEN)

token: fb5f5f596938e807

rank  score  category
   1     12  spoofing
   2     12  information disclosure
   3     12  denial of service
   4     10  tampering
   5     10  elevation of privilege
   6      6  repudiation

  highest score: spoofing at 12  (tied with information disclosure, denial of service)
  CHECK PASS: six risks scored and ranked
  token: fb5f5f596938e807


## Closing this session

Three `CHECK PASS` lines must appear above. If one does not, the fix is in the row that the
message names.

At minute 45, whichever platform you are on:

1. **Save the notebook into your repository.** In Colab: File, then Save a copy in GitHub,
   choose `mac237-labs`, set the path to
   `lab1/MAC237_LAB1_S01_Risk_And_Threat_Model.ipynb`, and write a commit message naming the
   session. On Kali: save the notebook into the `lab1` folder of your clone, then `git add -A`,
   `git commit`, `git push`.
2. Open the commit on github.com and confirm the notebook is actually there, with your token
   visible in the saved output. A save that failed silently looks exactly like one that worked
   until you look.
3. Screenshots, both showing your own username or Google account name and your token:
   `lab1_s01_repo_created.png`, the Settings, Collaborators page of your private repository
   with the instructor's invitation listed; and `lab1_s01_threat_model.png`, this notebook
   showing the three `CHECK PASS` lines and the ranked table.
4. Sign out before you leave and capture `lab1_s01_logout.png`. On Kali that is `gh auth
   logout`, or `git credential-cache exit` with the token deleted on GitHub. In a browser it
   is signing out of the Google account, plus Runtime, then Disconnect and delete runtime.

## What goes in the report

1. Your three CIA rows and your six STRIDE rows, as tables, with one short paragraph per CIA
   property explaining why the control you named addresses the threat you named.
2. Your ranked table, and then the argument the table cannot make for you: name the risk you
   would work on first, and say why it is or is not the one with the highest score. Cost,
   how quickly a control can be in place, and who has to agree are all legitimate reasons to
   depart from the ranking. Saying the number is highest is not an argument.
3. One risk you decided to accept rather than control, and the sentence you would say to the
   person who has to sign off on accepting it.
4. Cite CTPE Chapter 1 on security concepts.